# M03 · APIs, modelos locais e apps conversacionais: prática

Neste notebook vais:

1. Fazer chamadas com streaming e medir o time to first token.
2. Tratar erros com novas tentativas e backoff exponencial.
3. Correr um modelo local com Ollama e com transformers e comparar com um modelo na nuvem.
4. Usar LiteLLM para trocar de fornecedor com uma linha.
5. Implementar memória de conversa com janela deslizante e resumo.

**Requisitos:** `ANTHROPIC_API_KEY` no `.env` e [Ollama](https://ollama.com) instalado com `ollama pull qwen2.5:3b` (cerca de 2 GB).

Lição de apoio: [licao.html](licao.html)

In [ ]:
import os, time, random
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
MODELO = os.getenv("MODELO", "claude-haiku-5-5")
SYSTEM = "És o assistente da Loja Lusa, uma loja online portuguesa de artigos para casa. Responde em português de Portugal, de forma breve."

## 1. Streaming e time to first token

In [ ]:
def com_stream(pergunta, modelo=MODELO):
    t0 = time.time(); ttft = None; texto = ""
    with client.messages.stream(model=modelo, max_tokens=400, system=SYSTEM,
                                messages=[{"role": "user", "content": pergunta}]) as s:
        for pedaco in s.text_stream:
            if ttft is None:
                ttft = time.time() - t0
            texto += pedaco
            print(pedaco, end="", flush=True)
        uso = s.get_final_message().usage
    total = time.time() - t0
    print(f"\n\nTTFT {ttft:.2f}s | total {total:.2f}s | {uso.output_tokens / total:.0f} tokens/s")
    return texto

_ = com_stream("Sugere 5 ideias de presentes de Natal abaixo de 30 euros, com uma frase cada.")

## 2. Erros e novas tentativas

Vamos simular uma API instável e escrever um decorador com backoff exponencial e jitter. Depois fazemos o mesmo com `tenacity`.

In [ ]:
class ErroTemporario(Exception):
    pass

def api_instavel():
    if random.random() < 0.6:
        raise ErroTemporario("529 overloaded")
    return "ok"

def com_retry(f, tentativas=5, base=0.5):
    for i in range(tentativas):
        try:
            return f()
        except ErroTemporario as e:
            if i == tentativas - 1:
                raise
            espera = base * 2 ** i * (0.5 + random.random())   # backoff exponencial com jitter
            print(f"tentativa {i + 1} falhou ({e}), espero {espera:.2f}s")
            time.sleep(espera)

random.seed(3)
print(com_retry(api_instavel))

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential_jitter, retry_if_exception_type
import anthropic

@retry(stop=stop_after_attempt(5), wait=wait_exponential_jitter(initial=1, max=30),
       retry=retry_if_exception_type((anthropic.RateLimitError, anthropic.APIStatusError, anthropic.APIConnectionError)))
def chamar(msgs):
    return client.messages.create(model=MODELO, max_tokens=300, system=SYSTEM, messages=msgs, timeout=30)

print(chamar([{"role": "user", "content": "Qual é o prazo de devolução habitual em Portugal para compras online?"}]).content[0].text)

**Exercício 2.1.** O `APIStatusError` também apanha erros 400, que não vale a pena repetir. Escreve uma função `deve_repetir(exc)` que só devolve `True` para 429, 500, 502, 503, 529 e erros de ligação, e usa-a com `retry_if_exception(deve_repetir)`.

## 3. Modelos locais

### 3.1 Ollama

O Ollama expõe uma API compatível com a da OpenAI em `http://localhost:11434/v1`, por isso usamos o SDK da OpenAI só a mudar o `base_url`.

In [ ]:
from openai import OpenAI

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
t0 = time.time()
r = ollama.chat.completions.create(
    model="qwen2.5:3b",
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Fazem entregas na Madeira?"}],
)
print(r.choices[0].message.content, f"\n({time.time() - t0:.1f}s, {r.usage.completion_tokens} tokens)")

### 3.2 transformers

Com `transformers` tens o modelo em PyTorch dentro do processo. Mais lento sem GPU, mas dá acesso a tudo (logits, atenção, fine-tuning).

In [ ]:
import torch
from transformers import pipeline

gerador = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
                   device_map="auto", torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Fazem entregas na Madeira?"}]
print(gerador(msgs, max_new_tokens=80, do_sample=False)[0]["generated_text"][-1]["content"])

### 3.3 Comparação

Avalia os três modelos nas mesmas perguntas. Num projeto real usarias um conjunto de teste com respostas de referência; aqui lê as respostas e anota tu.

In [ ]:
perguntas = [
    "Posso devolver um artigo 20 dias depois de o receber?",
    "Qual a diferença entre um edredão de 250 e de 400 g/m²?",
    "Escreve um email curto a pedir desculpa por um atraso na entrega.",
]

def nuvem(p):
    return client.messages.create(model=MODELO, max_tokens=300, system=SYSTEM, messages=[{"role": "user", "content": p}]).content[0].text

def local(p):
    return ollama.chat.completions.create(model="qwen2.5:3b", messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}]).choices[0].message.content

linhas = []
for p in perguntas:
    for nome, f in [("nuvem", nuvem), ("local 3B", local)]:
        t0 = time.time(); resp = f(p)
        linhas.append({"pergunta": p[:40], "modelo": nome, "seg": round(time.time() - t0, 1), "resposta": resp})
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(linhas)

## 4. LiteLLM: uma chamada, vários fornecedores

In [ ]:
from litellm import completion, completion_cost

for modelo in [f"anthropic/{MODELO}", "ollama/qwen2.5:3b"]:
    t0 = time.time()
    r = completion(model=modelo, messages=[{"role": "user", "content": "Diz em 10 palavras o que é a Loja Lusa: loja online de artigos para casa."}])
    try:
        custo = completion_cost(r)
    except Exception:
        custo = 0.0
    print(f"{modelo:32} {time.time() - t0:5.1f}s  ${custo:.6f}  {r.choices[0].message.content}")

## 5. Memória de conversa

Implementa uma classe `Conversa` com duas estratégias: janela deslizante por tokens e resumo das mensagens antigas.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")
ntok = lambda m: len(enc.encode(m["content"]))

class Conversa:
    def __init__(self, estrategia="janela", limite_tokens=600, manter=4, modelo=f"anthropic/{MODELO}"):
        self.mensagens, self.resumo = [], ""
        self.estrategia, self.limite, self.manter, self.modelo = estrategia, limite_tokens, manter, modelo

    def _contexto(self):
        if self.estrategia == "completo":
            return self.mensagens
        if self.estrategia == "janela":
            ctx, total = [], 0
            for m in reversed(self.mensagens):
                total += ntok(m)
                if total > self.limite:
                    break
                ctx.insert(0, m)
            while ctx and ctx[0]["role"] != "user":   # a API exige começar com o utilizador
                ctx.pop(0)
            return ctx
        # resumo
        if sum(map(ntok, self.mensagens)) > self.limite and len(self.mensagens) > self.manter:
            antigas, self.mensagens = self.mensagens[:-self.manter], self.mensagens[-self.manter:]
            texto = "\n".join(f"{m['role']}: {m['content']}" for m in antigas)
            self.resumo = completion(model=self.modelo, messages=[{"role": "user", "content":
                f"Resumo anterior: {self.resumo or '(nenhum)'}\n\nNovas mensagens:\n{texto}\n\n"
                "Atualiza o resumo da conversa em no máximo 5 frases, mantendo nomes, números de encomenda, datas e decisões."}]
            ).choices[0].message.content
        return self.mensagens

    def enviar(self, texto):
        self.mensagens.append({"role": "user", "content": texto})
        system = SYSTEM + (f"\n\nResumo da conversa até agora: {self.resumo}" if self.resumo else "")
        ctx = self._contexto()
        r = completion(model=self.modelo, messages=[{"role": "system", "content": system}] + ctx, max_tokens=200)
        resposta = r.choices[0].message.content
        self.mensagens.append({"role": "assistant", "content": resposta})
        return resposta, r.usage.prompt_tokens

In [ ]:
guiao = [
    "Olá, sou a Marta e tenho a encomenda LL-20931 atrasada.",
    "Encomendei a 2 de novembro um conjunto de toalhas.",
    "Já agora, têm toalhas de praia?",
    "E de que cores?",
    "Qual é a política de devoluções?",
    "Ok. Voltando ao início: qual era o meu nome e o número da encomenda?",
]

for estrategia in ["completo", "janela", "resumo"]:
    c = Conversa(estrategia, limite_tokens=250)
    print(f"\n=== {estrategia} ===")
    for msg in guiao:
        resp, tin = c.enviar(msg)
    print(f"última chamada: {tin} tokens de entrada\n→ {resp}")

**Exercício 5.1.** Acrescenta uma estratégia `"factos"`: depois de cada resposta, pede ao modelo (com structured outputs, como no M02) uma lista de factos sobre o cliente e guarda-os num dicionário. Injeta os factos no system prompt. Compara com o resumo na última pergunta do guião.

## Use case: chatbot de apoio ao cliente

Em `use-case/` tens a Loja Lusa completa:

- `faq.md`: base de conhecimento da loja (entregas, devoluções, pagamentos).
- `app.py`: interface Gradio com streaming, memória por resumo e troca de modelo pela variável `MODELO_CHAT`.

```bash
cd modulos/03-apis-modelos-locais-e-apps-conversacionais/use-case
uv run python app.py                                   # modelo na nuvem
MODELO_CHAT=ollama/qwen2.5:7b uv run python app.py     # modelo local
```

**Desafio:** publica a app num Hugging Face Space (Gradio, gratuito) e põe o link no README do repositório.